In [12]:
from pathlib import Path
import os
from collections import defaultdict

from caf.base import DVector, ZoningSystem
from caf.base.zoning import TranslationWeighting
import pandas as pd

os.chdir(r'..\..')
from land_use.constants.geographies import CACHE_FOLDER, GORS, KNOWN_GEOGRAPHIES
from land_use.data_processing.mapping import create_interactive_maps
from land_use.data_processing import create_dvector_from_data, read_dvector_data, translate_and_combine_dvectors, generate_segment_bar_plots
os.chdir(r'supporting_processes\norcom')

In [2]:
# read in DVLA data and convert to DVector
dvla_data = Path(r"I:\NorMITs NorCOM\Import\DVLA\DVLA_vs_ONS_QA.xlsx")
dvla = pd.read_excel(dvla_data, sheet_name="Analysis")[["LSOA21CD", "DVLA 2023"]].dropna(subset="DVLA 2023")
dvla.columns = ["LSOA21CD", "cars"]
dvla["total"] = 1
pivoted = dvla.pivot_table(values="cars", columns="LSOA21CD", index="total")
dvla_dvec = create_dvector_from_data(
    dvector_data=pivoted, geographical_level="LSOA2021",
    input_segments=["total"]
)
# save outputs
dvla_dvec.save(dvla_data.parent / f"2023_dvla.hdf")
# dvla.to_hdf(dvla_data.parent / "2023_dvla.hdf", key="data", mode="w")

C:\ProgramData\Anaconda3\envs\normits_lu_jupyter\Lib\site-packages\caf\base\data_structures.py:443: UserWarning: 0.0 dropped on seg validation.
  warnings.warn(f"{full_sum - cut_sum} dropped on seg validation.")


In [3]:
for GOR in GORS:
    # create a dvector
    dvla_dvec = create_dvector_from_data(
        dvector_data=pivoted, geographical_level="LSOA2021",
        input_segments=["total"], geography_subset=GOR
    )
    # save regional outputs
    dvla_dvec.save(dvla_data.parent / f"2023_dvla_{GOR}.hdf")

The input data started with 35,672 columns. Filtering to NE results in 1,682 columns.
C:\ProgramData\Anaconda3\envs\normits_lu_jupyter\Lib\site-packages\caf\base\data_structures.py:443: UserWarning: 0.0 dropped on seg validation.
  warnings.warn(f"{full_sum - cut_sum} dropped on seg validation.")
The input data started with 35,672 columns. Filtering to NW results in 4,567 columns.
C:\ProgramData\Anaconda3\envs\normits_lu_jupyter\Lib\site-packages\caf\base\data_structures.py:443: UserWarning: 0.0 dropped on seg validation.
  warnings.warn(f"{full_sum - cut_sum} dropped on seg validation.")
The input data started with 35,672 columns. Filtering to YH results in 3,355 columns.
C:\ProgramData\Anaconda3\envs\normits_lu_jupyter\Lib\site-packages\caf\base\data_structures.py:443: UserWarning: 0.0 dropped on seg validation.
  warnings.warn(f"{full_sum - cut_sum} dropped on seg validation.")
The input data started with 35,672 columns. Filtering to Wales results in 1,917 columns.
C:\ProgramData\An

In [4]:
# convert the "car availability" DVectors output from the model to an estimate of total cars
_2_plus_cars = 2.5
car_converter = pd.DataFrame({
    "car_availability": [1, 2, 3],
    "W92000004": [0, 1, _2_plus_cars],
    "E12000008": [0, 1, _2_plus_cars],
    "E12000004": [0, 1, _2_plus_cars],
    "E12000005": [0, 1, _2_plus_cars],
    "E12000002": [0, 1, _2_plus_cars],
    "E12000009": [0, 1, _2_plus_cars],
    "E12000007": [0, 1, _2_plus_cars],
    "E12000003": [0, 1, _2_plus_cars],
    "E12000001": [0, 1, _2_plus_cars],
    "E12000006": [0, 1, _2_plus_cars],
}).set_index(['car_availability'])
car_converter.columns.name = "RGN2021"
car_converter

RGN2021,W92000004,E12000008,E12000004,E12000005,E12000002,E12000009,E12000007,E12000003,E12000001,E12000006
car_availability,,,,,,,,,,
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
3,2.5,2.5,2.5,2.5,2.5,2.5,2.5,2.5,2.5,2.5


In [5]:
car_converter_dvec = create_dvector_from_data(
    dvector_data=car_converter, geographical_level="RGN2021",
    input_segments=["car_availability"]
)
# convert to LSOA
car_converter_lsoa = car_converter_dvec.translate_zoning(
    new_zoning=KNOWN_GEOGRAPHIES.get(f'LSOA2021'),
    cache_path=CACHE_FOLDER, weighting=TranslationWeighting.NO_WEIGHT
)
car_converter_lsoa.data

C:\ProgramData\Anaconda3\envs\normits_lu_jupyter\Lib\site-packages\caf\base\data_structures.py:443: UserWarning: 0.0 dropped on seg validation.
  warnings.warn(f"{full_sum - cut_sum} dropped on seg validation.")
C:\ProgramData\Anaconda3\envs\normits_lu_jupyter\Lib\site-packages\caf\base\zoning.py:681: TranslationWarning: 10 RGN2021 zones have splitting factors which don't sum to 1 (value totals may change during zone_translation), the maximum difference is 5.6e+03
  warnings.warn(
C:\ProgramData\Anaconda3\envs\normits_lu_jupyter\Lib\site-packages\caf\toolkit\translation.py:906: UserWarning: Some values seem to have been dropped. The difference total is 124817.0 (translated - original).
  warnings.warn(
C:\ProgramData\Anaconda3\envs\normits_lu_jupyter\Lib\site-packages\caf\base\data_structures.py:443: UserWarning: 0.0 dropped on seg validation.
  warnings.warn(f"{full_sum - cut_sum} dropped on seg validation.")


lsoa2021_id,E01000001,E01000002,E01000003,E01000005,E01000006,E01000007,E01000008,E01000009,E01000011,E01000012,...,W01002031,W01002032,W01002033,W01002034,W01002035,W01002036,W01002037,W01002038,W01002039,W01002040
car_availability,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
3,2.5,2.5,2.5,2.5,2.5,2.5,2.5,2.5,2.5,2.5,...,2.5,2.5,2.5,2.5,2.5,2.5,2.5,2.5,2.5,2.5


In [6]:
# save output DVector and read back in / save as gor specific ones
car_converter_lsoa.save(dvla_data.parent / "car_converters" / f"car_converter.hdf")
for GOR in GORS:
    data = read_dvector_data(
        dvla_data.parent / "car_converters" / f"car_converter.hdf",
        geographical_level="LSOA2021", input_segments=['car_availability'], geography_subset=GOR, hdf_key="data"
    )
    data.save(dvla_data.parent / "car_converters" / f"car_converter_{GOR}.hdf")

The input data at I:\NorMITs NorCOM\Import\DVLA\car_converters\car_converter.hdf started with 35,672 columns. Filtering to NE results in 1,682 columns.
C:\ProgramData\Anaconda3\envs\normits_lu_jupyter\Lib\site-packages\caf\base\data_structures.py:443: UserWarning: 0.0 dropped on seg validation.
  warnings.warn(f"{full_sum - cut_sum} dropped on seg validation.")
The input data at I:\NorMITs NorCOM\Import\DVLA\car_converters\car_converter.hdf started with 35,672 columns. Filtering to NW results in 4,567 columns.
C:\ProgramData\Anaconda3\envs\normits_lu_jupyter\Lib\site-packages\caf\base\data_structures.py:443: UserWarning: 0.0 dropped on seg validation.
  warnings.warn(f"{full_sum - cut_sum} dropped on seg validation.")
The input data at I:\NorMITs NorCOM\Import\DVLA\car_converters\car_converter.hdf started with 35,672 columns. Filtering to YH results in 3,355 columns.
C:\ProgramData\Anaconda3\envs\normits_lu_jupyter\Lib\site-packages\caf\base\data_structures.py:443: UserWarning: 0.0 dro

In [14]:
# define pathto NorCOM validation outputs
OUTPUT_DIR = Path(r'I:\NorMITs NorCOM\Validation\v38\2023\data')
RESULTS_DIR = Path(r'I:\NorMITs NorCOM\Validation\v38\2023\maps')
RESULTS_DIR.mkdir(exist_ok=True)

In [8]:
# read in NorCOM outputs (households with number of cars) and convert to approx number of cars
for GOR in GORS:
    modelled = DVector.load(OUTPUT_DIR / f'Expected_{GOR}.hdf')
    car_converter = DVector.load(dvla_data.parent / "car_converters" / f"car_converter_{GOR}.hdf")
    approx_cars = modelled * car_converter
    approx_cars = approx_cars.add_segments(['total']).aggregate(['total'])
    approx_cars.save(OUTPUT_DIR / f'Approx_Cars_{GOR}.hdf')

In [15]:
# create lists of data to use
total_file_dict = defaultdict(list)
northern_file_dict = defaultdict(list)
# get files from existing output
total_file_dict['DVLA'].extend(dvla_data.parent.glob('2023_dvla_*.hdf'))
total_file_dict['Approximated'].extend(OUTPUT_DIR.glob('Approx_Cars_*.hdf'))
northern_file_dict['DVLA'].extend([dvla_data.parent / "2023_dvla_NW.hdf", dvla_data.parent / "2023_dvla_NE.hdf", dvla_data.parent / "2023_dvla_YH.hdf"])
northern_file_dict['Approximated'].extend([OUTPUT_DIR / "Approx_Cars_NW.hdf", OUTPUT_DIR / "Approx_Cars_NE.hdf", OUTPUT_DIR / "Approx_Cars_YH.hdf"])        

In [ ]:
MAPPING_INPUTS = {
    'LAD2021+SCOTLANDLAD': total_file_dict,
    'MSOA21-NORTH': northern_file_dict
}
for MAP_ZONE_SYSTEM, file_dict in MAPPING_INPUTS.items():
    # Calculate all of our "total" dictionaries in one go
    data_dict = {}
    for key, input_files in file_dict.items():
        print(key)
        if not input_files:
            continue
        
        data_dict[key] = translate_and_combine_dvectors(
            input_files=input_files,
            aggregate_zone_system=MAP_ZONE_SYSTEM
        )
    for unit, map_total_dvector in data_dict.items():
        # Set up the output directory for that unit category
        results_dir = RESULTS_DIR / MAP_ZONE_SYSTEM / unit
        results_dir.mkdir(exist_ok=True, parents=True)
    
        # chart_total_dvector = map_total_dvector.translate_zoning(
        #     ZoningSystem.get_zoning(CHART_ZONE_SYSTEM, search_dir=CACHE_FOLDER)
        # )
    
        # Store all segment names so we can figure out which ones we skip
        all_segment_names = set(map_total_dvector.segmentation.names)
    
        for segment in all_segment_names:
    
            # Make the maps
            map_paths = create_interactive_maps(
                map_total_dvector, output_folder=results_dir, 
                specific_segment=segment,
                # filter_by={'RGN21CD': ['E12000001', 'E12000002', 'E12000003']},
                # filter_by={'LAD21CD': [
                #     'E08000009', 'E08000005', 'E08000037', 'E06000009', 'E08000017', 'E06000004','E08000015','E08000023',
                #     'E06000008', 'E07000167', 'E06000003', 'E06000049', 'E08000032', 'E07000120', 'E06000047', 'E07000163',
                #     'E08000012', 'E07000027', 'E08000006', 'E07000119', 'E06000007', 'E06000014', 'E08000008', 'E06000002',
                #     'E07000030', 'E07000122', 'E06000013', 'E06000050', 'E06000005', 'E08000010', 'E07000031', 'E08000007',
                #     'E08000022', 'E06000011', 'E07000126', 'E07000124', 'E07000026', 'E07000028', 'E07000164', 'E07000117',
                #     'E07000168', 'E08000018', 'E06000057', 'E07000169', 'E08000016', 'E08000034', 'E07000127', 'E06000012',
                #     'E08000003', 'E07000121', 'E08000013', 'E08000033', 'E07000029', 'E06000001', 'E07000128', 'E08000014',
                #     'E07000165', 'E07000123', 'E07000118', 'E08000024', 'E08000011', 'E08000004', 'E06000006', 'E06000010',
                #     'E07000125', 'E08000021', 'E08000036', 'E08000035', 'E07000166', 'E08000019', 'E08000002', 'E08000001'
                # ]},
                filter_by=None,
                simplification=100,
                fix_proportional_scale=True
            )

In [16]:
# Calculate all of our "total" dictionaries in one go
data_dict = {}
for key, input_files in total_file_dict.items():
    print(key)
    if not input_files:
        continue
    
    data_dict[key] = translate_and_combine_dvectors(
        input_files=input_files,
        aggregate_zone_system='LAD2021+SCOTLANDLAD'
    )
for unit, map_total_dvector in data_dict.items():
    # Set up the output directory for that unit category
    results_dir = RESULTS_DIR / 'LAD2021+SCOTLANDLAD' / unit
    results_dir.mkdir(exist_ok=True, parents=True)

    chart_total_dvector = map_total_dvector.translate_zoning(
        ZoningSystem.get_zoning('LAD2021+SCOTLANDLAD', search_dir=CACHE_FOLDER)
    )

    # Store all segment names so we can figure out which ones we skip
    all_segment_names = set(chart_total_dvector.segmentation.names)

    for segment_plot in generate_segment_bar_plots(chart_total_dvector, unit=unit):
        # First - save the figure
        segment_plot.figure.savefig(results_dir / f'{segment_plot.segments}.png')

        # And save the data
        segment_plot.summary_data.to_csv(
            results_dir / f'{segment_plot.segments}.csv', 
            float_format=lambda x: '{:,.0f}'.format(x)
        )
    

DVLA
Approximated
